## Importing packages

In [475]:
# Import packages
import pandas as pd
import numpy as np
import re

import altair as alt
import pandas as pd

import warnings

from sklearn.linear_model import LinearRegression

import statsmodels.formula.api as smf

import plotly.graph_objects as go

In [210]:
warnings.filterwarnings('ignore')

## Importing data and preprocesing

In [211]:
# Import price data
prices_25 = pd.read_csv('/Users/sambickel-barlow/Desktop/PP422/Final Project/prices_2025_cross_section.csv')
# Import items data
items_25 = pd.read_csv('/Users/sambickel-barlow/Desktop/PP422/Final Project/items_2025.csv')
# Import categorized and gender labeled product data
items_cat_25 = pd.read_csv('/Users/sambickel-barlow/Desktop/PP422/Final Project/absolute_final_classifier.csv')


In [212]:
# Merge product url from items_25 to prices data to avoid bad merge on product_id
prices_25_2 = prices_25.merge(items_25[['store_id','product_id','product_url']], how='inner',on=['store_id','product_id'])

In [213]:
# Merge price data with labeled product data
items_prices = items_cat_25[['product_url','category','final_classifier']].merge(prices_25_2[['store_id','product_id','product_url','price','unit_price','loyalty_price','original_price']],how='left',on=['product_url'])

In [229]:
# Remove Adult Intimacy and Vape categories
items_prices = items_prices[(items_prices['category'] != 'Adult Intimacy') & (items_prices['category'] != 'Vape')]

In [230]:
# Mapping for the 'final_classifier' column
gender_map = {0: 'Female', 1: 'Male', 2: 'Neutral'}

# Create a new column 'gender' based on 'final_classifier'
items_prices['gender'] = items_prices['final_classifier'].map(gender_map)

In [476]:
# Using Ignacio's code to standardize unit prices across for each unit type
merged_df_prices = items_prices
merged_df_prices[['price_per_unit', 'unit']] = merged_df_prices['unit_price'].str.split('/', expand=True) # Splitting prices

merged_df_prices['price_per_unit'] = merged_df_prices['price_per_unit'].str.replace("£", "").str.replace(" ", "") # Replace these characters
merged_df_prices['price_per_unit'] = pd.to_numeric(merged_df_prices['price_per_unit'], errors='coerce') # To numeric variable

merged_df_prices['unit'] = merged_df_prices['unit'].str.replace(" ", "") # Replace these characters
merged_df_prices['unit'].unique()

array(['100ml', 'each', 'litre', 'kg', '100g', '100sht', 'metre', '10ml',
       'ltr', 'ea', 'wipe', nan, 'lt', '10g', 'mr', None, '100sheets'],
      dtype=object)

In [477]:
# Calculate loyalty and normal discounts
merged_df_prices['loyalty_discount'] = (merged_df_prices['price'] - merged_df_prices['loyalty_price']) / merged_df_prices['price']
merged_df_prices['normal_discount'] = (merged_df_prices['original_price'] - merged_df_prices['price']) / merged_df_prices['original_price']

In [478]:
# Create a single column for all discounts
merged_df_prices['discount'] = np.where(
    merged_df_prices[['loyalty_discount', 'normal_discount']].isna().all(axis=1),  # If both are NA
    0,  
    np.where(
        merged_df_prices['loyalty_discount'].isna(),  # If col1 is NA, take col2
        merged_df_prices['normal_discount'],
        np.where(
            merged_df_prices['normal_discount'].isna(),  # If col2 is NA, take col1
            merged_df_prices['loyalty_discount'],
            merged_df_prices[['loyalty_discount', 'normal_discount']].max(axis=1)  # If both exist, take the larger value
        )
    )
)

In [479]:
# Using Ignacio's code to standardize unit prices across for each unit type
merged_df_prices_unit = merged_df_prices

# 0) NaN
merged_df_prices_unit['unit'] = merged_df_prices_unit['unit'].replace(['None', np.nan], np.nan)

# 1)  "Unit" category
# 2 each
merged_df_prices_unit.loc[merged_df_prices_unit['unit'].isin(['2each','pair']), 'price_per_unit'] *= 0.5

# 10 each
merged_df_prices_unit.loc[merged_df_prices_unit['unit'].isin(['10each']), 'price_per_unit'] *= 0.1

# 50 each
merged_df_prices_unit.loc[merged_df_prices_unit['unit'].isin(['50each']), 'price_per_unit'] *= 0.02

# 100 each
merged_df_prices_unit.loc[merged_df_prices_unit['unit'].isin(['100ea']), 'price_per_unit'] *= 0.01
     
#merged_df_prices_unit.loc[merged_df_prices_unit['unit'].str.contains('each', na=False), 'unit_price'] = 'unit'

# 2) "kg" category

# 100g
merged_df_prices_unit.loc[merged_df_prices_unit['unit'].isin(['100g','100gdrained','100gDR.WT']), 'price_per_unit'] *= 10
# 317g 
merged_df_prices_unit.loc[merged_df_prices_unit['unit'].isin(['317g']), 'price_per_unit'] *= 1000/317
# 10g
merged_df_prices_unit.loc[merged_df_prices_unit['unit'].isin(['10g','10gDR.WT']), 'price_per_unit'] *= 100

# 3) "Liters" category
# 100ml
merged_df_prices_unit.loc[merged_df_prices_unit['unit'].isin(['100ml','100mlDR.WT']), 'price_per_unit'] *= 10
# 75cl 
merged_df_prices_unit.loc[merged_df_prices_unit['unit'].isin(['75cl','75c3']), 'price_per_unit'] *= 1000/750
# 10ml
merged_df_prices_unit.loc[merged_df_prices_unit['unit'].isin(['10ml']), 'price_per_unit'] *= 100
# 1ml
merged_df_prices_unit.loc[merged_df_prices_unit['unit'].isin(['ml']), 'price_per_unit'] *= 1000

# 4) "Metres" category
# Changing 'units' variable to lt
merged_df_prices_unit['unit'] = merged_df_prices_unit['unit'].replace(['metre','mr','mtr'], 'mt')

# 5) "Sheets" category
# 100 sheets 
merged_df_prices_unit.loc[merged_df_prices_unit['unit'].isin(['100sheets','100sht']), 'price_per_unit'] *= 0.01
# 10 sheets 
merged_df_prices_unit.loc[merged_df_prices_unit['unit'].isin(['10sheets']), 'price_per_unit'] *= 0.1


# Changing the 'units' column to unit         
merged_df_prices_unit['unit'] = merged_df_prices_unit['unit'].replace(['each', 'ea','2each','pair','10each','50each','100ea','wipe','bisc','batt'], 'unit')

# Changing 'units' column to kg
merged_df_prices_unit['unit'] = merged_df_prices_unit['unit'].replace(['100g', '100gdrained','100gDR.WT','317g','10g','10gDR.WT','kgDR.WT','kgdrained'], 'kg')

# Changing 'units' column to lt
merged_df_prices_unit['unit'] = merged_df_prices_unit['unit'].replace(['100ml', '100mlDR.WT','75cl','75c3','10ml','ml','litre','ltr','litreDR.WT'], 'lt')

# Changing 'units' column to sheet
merged_df_prices_unit['unit'] = merged_df_prices_unit['unit'].replace(['100sheets','100sht','10sheets'], 'sheet')

# Changing some rare cases (where I still have missing values in price_per_unit variable)
merged_df_prices_unit['price_per_unit'] = merged_df_prices_unit['price_per_unit'].fillna(merged_df_prices_unit['unit_price'].str.extract(r'(\d+(\.\d+)?)p')[0]) # Fill this missings with characters before the first 'p'
merged_df_prices_unit['price_per_unit'] = merged_df_prices_unit['price_per_unit'].fillna(merged_df_prices_unit['unit_price'].str.extract(r'£(\d+(\.\d+)?)')[0]) # Fill this missings with characters after £ and before first space
merged_df_prices_unit = merged_df_prices_unit.reset_index(drop=True)
merged_df_prices_unit.loc[merged_df_prices_unit['unit_price'].str.contains(r'\(per kg\)', na=False), 'price_per_unit'] = merged_df_prices_unit['price']

merged_df_prices_unit['price_per_unit'] = merged_df_prices_unit['price_per_unit'].astype(float) # Convert the extracted prices to float

merged_df_prices_unit.loc[merged_df_prices_unit['unit_price'].str.contains(r'each', na=False), 'unit'] = 'unit' # Fill with 'unit' if the variable contains 'each'
merged_df_prices_unit.loc[merged_df_prices_unit['unit_price'].str.contains(r'\(per kg\)', na=False), 'unit'] = 'kg' # Fill with 'kg' if the variable contains '(per kg)'

In [481]:
# Remove rows missing unit price
merged_df_prices_unit = merged_df_prices_unit[~merged_df_prices_unit['unit'].isna()]

In [482]:
merged_df_prices_unit.shape

(5410, 16)

## Visualizations and Regressions

In [483]:
alt.data_transformers.enable('default', max_rows=6000)  # Remove row limit

DataTransformerRegistry.enable('default')

In [484]:
# Function to remove outliers for visualisations based on log-transformed data
def remove_log_outliers(df, column):
    df['log_price_per_unit'] = np.log1p(df[column])  # Apply log transformation
    Q1 = df['log_price_per_unit'].quantile(0.25)
    Q3 = df['log_price_per_unit'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df['log_price_per_unit'] >= lower_bound) & (df['log_price_per_unit'] <= upper_bound)]

# Remove log-transformed outliers
df_filtered = remove_log_outliers(merged_df_prices_unit, 'price_per_unit')

# Convert to Altair-friendly format
df_filtered = df_filtered.copy()

### Scatterplot for Selected Categories

In [462]:
# Select product categories to visualize
df_filtered_select = df_filtered[df_filtered['category'].isin(['Oral Care','Deodorant','Body Wash','Shampoo/Conditioner'])]

# Custom color scale for gender
color_scale = alt.Scale(domain=['Female', 'Male', 'Neutral'], range=['#ff69b4', '#1f77b4', '#2ca02c'])  # Pink, Blue, Green

# Base scatter plot with semi-transparent circles
points = alt.Chart(df_filtered_select).mark_circle(opacity=0.5).encode(
    y=alt.Y('gender:N', title=None),  # No y-axis title
    x=alt.X('log_price_per_unit:Q', title='Log(Price Per Unit)',  
            axis=alt.Axis(titleFontSize=16)),  # Increase x-axis title font size
    color=alt.Color('gender:N', scale=color_scale, legend=alt.Legend(title="Gender")),
    tooltip=['store_id','category', 'price_per_unit', 'unit'],
).properties(
    width=600,
    height=100
)

# Compute mean price in log scale per (category, gender)
mean_marks = alt.Chart(df_filtered_select ).transform_joinaggregate(
    mean_price='mean(log_price_per_unit)', 
    groupby=['category', 'gender']
).mark_text(
    text='×',
    size=16,
    color='black'
).encode(
    y='gender:N',  
    x='mean_price:Q'
)

# Create the faceted chart
scatter_chart = (points + mean_marks).facet(
    row=alt.Row('category:N', 
               header=alt.Header(
                   title=None,  # Remove "category" title
                   labelOrient='top',
                   labelPadding=5,
                   labelFontSize=16))  # Increase category label font size
).properties(
    title='Price Distribution by Product Category and Gender'
)

# Annotation as a simple chart below
annotation = alt.Chart(pd.DataFrame({'text': ["\"×\" marks the mean log(price)"]})).mark_text(
    align='left',
    baseline='top',
    fontSize=14
).encode(
    text='text'
).properties(
    width=600,
    height=30
)

# Combine scatter plot and annotation
final_chart = alt.vconcat(
    scatter_chart,
    annotation,
    spacing=5
).configure_title(
    anchor='start',  # Left-align the title
    fontSize=16,
    offset=20
).configure_header(
    labelOrient='top'
).configure_view(
    stroke=None
)

final_chart


alt.VConcatChart(...)

### Density plot for Selected Categories

In [442]:
# Select product categories to visualize
df_filtered_select = df_filtered[df_filtered['category'].isin(['Shaving Cream', 'Skincare','Supplements'])]

# Custom color scale for gender
color_scale = alt.Scale(domain=['Female', 'Male', 'Neutral'], 
                       range=['#ff69b4', '#1f77b4', '#2ca02c'])  # Pink, Blue, Green

# Create density plots with transparency
density_plots = alt.Chart(df_filtered_select).transform_density(
    'log_price_per_unit',
    groupby=['category', 'gender'],
    as_=['log_price_per_unit', 'density'],
    extent=[df_filtered_select['log_price_per_unit'].min(), df_filtered_select['log_price_per_unit'].max()],
    bandwidth=0.1  # Adjust this for smoother/rougher distributions
).mark_area(
    opacity=0.5,  # Set transparency for overlap visibility
    interpolate='monotone',
    stroke='white',
    strokeWidth=0.5
).encode(
    x=alt.X('log_price_per_unit:Q', title='Log(1 + Price Per Unit)'),
    y=alt.Y('density:Q', title='Density', stack=None),  # stack=None allows overlaps
    color=alt.Color('gender:N', scale=color_scale, legend=alt.Legend(title="Gender")),
    tooltip=['category', 'gender']
).properties(
    width=600,
    height=150
)

# Add vertical lines for mean values
mean_lines = alt.Chart(df_filtered_select).transform_aggregate(
    mean_price='mean(log_price_per_unit)',
    groupby=['category', 'gender']
).mark_rule(
    strokeDash=[3,3],
    size=2
).encode(
    x='mean_price:Q',
    color='gender:N',
    opacity=alt.value(0.8)
)

# Combine and facet by category
density_chart = (density_plots + mean_lines).facet(
    row=alt.Row('category:N', 
               header=alt.Header(
                   title=None,
                   labelOrient='top',
                   labelPadding=5 ,
                   labelFontSize=16))
).properties(
    title='Price Distribution Density by Product Category and Gender'
).configure_title(
    anchor='start',
    fontSize=16,
    offset=20
).configure_header(
    labelOrient='top'
).configure_view(
    stroke=None
)

density_chart

alt.FacetChart(...)

### Boxplot for Selected Categories

In [272]:
# Prepare the data
df_filtered['gender'] = df_filtered['gender'].astype(str)
df_filtered['log_price_per_unit'] = df_filtered['log_price_per_unit'].astype(float)

df_filtered_select = df_filtered[df_filtered['category'].isin(['Body Wash', 'Shampoo/Conditioner','Skincare','Deodorant'])]

# Custom color scale
color_scale = alt.Scale(domain=['Female', 'Male', 'Neutral'], 
                       range=['#ff69b4', '#1f77b4', '#2ca02c'])

# Main box plot with means
main_chart = alt.Chart(df_filtered_select).mark_boxplot(
    extent=0.5,
    size=20,
    opacity=0.8
).encode(
    x=alt.X('log_price_per_unit:Q', title='Log(1 + Price Per Unit)'),
    y=alt.Y('gender:N', title=None, axis=alt.Axis(ticks=False)),
    color=alt.Color('gender:N', scale=color_scale, legend=None)
).properties(
    width=500,
    height=75
)

# Add mean markers
mean_points = alt.Chart(df_filtered_select).mark_point(
    shape='diamond',
    size=50,
    color='black'
).encode(
    x='mean(log_price_per_unit):Q',
    y='gender:N'
)

# Create the faceted chart
faceted_chart = (main_chart + mean_points).facet(
    row=alt.Row('category:N', 
               header=alt.Header(title=None, labelOrient='top', labelFontSize=15))
)

# Create legend separately
legend = alt.Chart(df_filtered_select).mark_point().encode(
    color=alt.Color('gender:N', scale=color_scale, 
                  legend=alt.Legend(title="Gender"))
).properties(
    width=100
)

# Combine with proper configuration
box_chart = alt.hconcat(
    faceted_chart,
    legend
).properties(
    title='Price Distribution Comparison by Product Category and Gender (Selected Categories)'
).configure_title(
    anchor='start',
    fontSize=16,
    offset=20
).configure_view(
    stroke=None
).configure_axis(
    grid=False,
    labelFontSize=13  # Increases y-axis label size
)

box_chart

alt.HConcatChart(...)

### Discount plots

#### Discount by category

In [486]:
# Calculate both mean discount and count of discounted items
discount_stats = (merged_df_prices[merged_df_prices['discount'] > 0]
                 .groupby(['gender','category'])
                 .agg(mean_discount=('discount', 'mean'),
                      count=('discount', 'count'))
                 .reset_index())

In [487]:
# Define custom colors for gender
gender_colors = {"Female": '#ff69b4', "Male": "#1f77b4", "Neutral": "#2ca02c"}

# Create the grouped bar chart with count labels
bars = alt.Chart(discount_stats).mark_bar().encode(
    x=alt.X("category:N", axis=alt.Axis(title=None, labelAngle=-45)),
    y=alt.Y("mean_discount:Q", title="Average Discount", axis=alt.Axis(format=".0%", titleAngle=0, titleY=-10)),
    color=alt.Color("gender:N", 
                  scale=alt.Scale(domain=list(gender_colors.keys()), 
                                range=list(gender_colors.values())),
    legend=alt.Legend(title="Gender")),
    xOffset="gender:N"
).properties(
    width=600,
    title={
        "text": "Average Discounts by Category and Gender",
    }
)

# Add count labels above bars
count_labels = alt.Chart(discount_stats).mark_text(
    align='center',
    baseline='bottom',
    dy=-5,  # Adjust this to position the label above the bar
    fontSize=9
).encode(
    x=alt.X("category:N"),
    y="mean_discount:Q",
    xOffset="gender:N",
    text=alt.Text("count:Q", format=".0f"),
    color=alt.value("black")
)

# Add explanatory annotation
annotation = alt.Chart(pd.DataFrame({'text': ['Note: Numbers above bars show sample sizes']})).mark_text(
    align='left',
    baseline='top',
    fontSize=10,
    color='gray'
).encode(
    text='text:N'
).properties(
    width=600
)

# Combine all elements
chart = alt.vconcat(
    bars + count_labels,
    annotation
).configure_view(
    stroke='transparent'
).configure_axis(
    grid=False
).configure_title(
    anchor='middle',
    subtitleFontSize=11,
    subtitlePadding=5
)

chart

alt.VConcatChart(...)

#### Discount by type

In [382]:
# Calculate both mean discount and count of discounted items
discount_stats2 = (merged_df_prices[merged_df_prices['discount'] > 0]
                 .groupby(['gender'])
                 .agg(mean_discount=('discount', 'mean'),
                      mean_ldiscount=('loyalty_discount', 'mean'),
                      mean_ndiscount=('normal_discount', 'mean'),
                      count=('discount', 'count'),
                      lcount=('loyalty_discount', 'count'),
                      ncount=('normal_discount', 'count'))
                 .reset_index())

discount_a = discount_stats2[['gender','mean_discount','mean_ldiscount','mean_ndiscount']].melt(id_vars='gender', 
                         var_name='Discount Type', 
                         value_name='Discount Value')
discount_count = discount_stats2[['gender','count','lcount','ncount']].melt(id_vars='gender', 
                         var_name='Discount Type', 
                         value_name='Discount Count')
discounts = pd.concat([discount_a,discount_count],axis=1).iloc[:,2:]

In [428]:
discounts['Discount Type'] = discounts['Discount Type'].replace({
    'count': 'Overall Discount',
    'lcount': 'Loyalty Discount',
    'ncount': 'Normal Discount'
})

# Define custom colors for gender
gender_colors = {"Female": '#ff69b4', "Male": "#1f77b4", "Neutral": "#2ca02c"}

# Create the grouped bar chart with count labels
bars = alt.Chart(discounts).mark_bar().encode(
    x=alt.X("Discount Type:N", axis=alt.Axis(title=None, labelAngle=0)),
    y=alt.Y("Discount Value:Q", title="Average Discount", axis=alt.Axis(format=".0%", titleAngle=0, titleY=-10)),
    color=alt.Color("gender:N", 
                  scale=alt.Scale(domain=list(gender_colors.keys()), 
                                range=list(gender_colors.values())),
    legend=alt.Legend(title="Gender")),
    xOffset="gender:N"
).properties(
    width=600,
    title={
        "text": "Average Discounts by Type and Gender",
    }
)

# Add count labels above bars
count_labels = alt.Chart(discounts).mark_text(
    align='center',
    baseline='bottom',
    dy=-5,  # Adjust this to position the label above the bar
    fontSize=9
).encode(
    x=alt.X("Discount Type:N"),
    y="Discount Value:Q",
    xOffset="gender:N",
    text=alt.Text("Discount Count:Q", format=".0f"),
    color=alt.value("black")
)

# Add explanatory annotation
annotation = alt.Chart(pd.DataFrame({'text': ['Note: Numbers above bars show sample sizes']})).mark_text(
    align='left',
    baseline='top',
    fontSize=10,
    color='gray'
).encode(
    text='text:N'
).properties(
    width=600
)

# Combine all elements
chart = alt.vconcat(
    bars + count_labels,
    annotation
).configure_view(
    stroke='transparent'
).configure_axis(
    grid=False
).configure_title(
    anchor='middle',
    subtitleFontSize=11,
    subtitlePadding=5
)

chart

alt.VConcatChart(...)

### Regressions and Coefficient plots

#### Regress gender on log price controling for category

In [273]:
reg_lup_cat = smf.ols('log_price_per_unit ~ gender + category' , data=merged_df_prices_unit).fit()
print(reg_lup_cat.summary())

                            OLS Regression Results                            
Dep. Variable:     log_price_per_unit   R-squared:                       0.253
Model:                            OLS   Adj. R-squared:                  0.251
Method:                 Least Squares   F-statistic:                     152.2
Date:                Sat, 29 Mar 2025   Prob (F-statistic):               0.00
Time:                        18:32:09   Log-Likelihood:                -8333.3
No. Observations:                5410   AIC:                         1.669e+04
Df Residuals:                    5397   BIC:                         1.678e+04
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

#### Coeffficient plot of above regression

In [488]:
# Extract coefficients and standard errors
coefs = reg_lup_cat.params
errors = reg_lup_cat.bse

# Filter just the gender coefficients
gender_terms = coefs[coefs.index.str.contains("gender")]
gender_df = pd.DataFrame({
    "Variable": gender_terms.index,
    "Coefficient": gender_terms.values,
    "SE": errors[gender_terms.index]
})

# Clean gender labels
gender_df["Gender"] = gender_df["Variable"].str.extract(r'gender\[T\.(.*?)\]')[0]

# Calculate confidence intervals
gender_df["CI_lower"] = gender_df["Coefficient"] - 1.96*gender_df["SE"]
gender_df["CI_upper"] = gender_df["Coefficient"] + 1.96*gender_df["SE"]

# Sort by coefficient value
gender_df = gender_df.sort_values("Coefficient")

# Create alternating background colors
n_categories = len(gender_df)
background_colors = ['#f9f9f9','#ededed'] * ((n_categories + 1) // 2)

# Create figure
fig = go.Figure()

# Add alternating background rectangles
for i, row in enumerate(gender_df.itertuples()):
    fig.add_shape(
        type="rect",
        x0=-max(abs(gender_df["CI_lower"].max()), abs(gender_df["CI_upper"].max())) - 0.5,
        x1=max(abs(gender_df["CI_lower"].max()), abs(gender_df["CI_upper"].max())) + 0.5,
        y0=i-0.5, y1=i+0.5,
        fillcolor=background_colors[i],
        layer="below",
        line_width=0
    )

# Add coefficient points
fig.add_trace(go.Scatter(
    x=gender_df["Coefficient"],
    y=gender_df.index,
    mode="markers",
    marker=dict(
        color=np.where(gender_df["Gender"] == "Male", "#1f77b4", "#2ca02c"),
        size=18,
        line=dict(width=1, color="black")
    ),
    error_x=dict(
        array=gender_df["SE"],
        color="rgba(0,0,0,0.7)",
        thickness=1.5,
        width=5
    ),
    text=gender_df["Gender"],
    textposition="middle left",
    textfont=dict(size=14),
    customdata=np.stack([
        gender_df["Coefficient"],
        gender_df["CI_lower"],
        gender_df["CI_upper"]
    ], axis=-1),
    hovertemplate=(
        "<b>%{text}</b><br>"
        "Estimate: %{customdata[0]:.2f}<br>"
        "95% CI: [%{customdata[1]:.2f}, %{customdata[2]:.2f}]<extra></extra>"
    )
))

# Add vertical zero line
fig.add_vline(
    x=0, 
    line_width=1.5, 
    line_color="red", 
    opacity=0.8,
    layer="above"
)

# Customize layout
fig.update_layout(
    title=dict(
        text="Gender Coefficient Estimates",
        font=dict(size=18),
        x=0.1,  # Align to the left
        xanchor="left"  # Anchor the title to the left
    ),
    xaxis=dict(
        title="Coefficient Value",
        titlefont=dict(size=14),
        showgrid=False,
        zeroline=False,
        tickfont=dict(size=12)
    ),
    yaxis=dict(
        title="",
        showgrid=False,
        zeroline=False,
        tickvals=gender_df.index,
        ticktext=gender_df["Gender"],
        tickfont=dict(size=14),
        automargin=True
    ),
    margin=dict(l=100, r=50, t=80, b=60),
    height=400,
    width=700,
    plot_bgcolor="white",
    showlegend=False
)

fig.show()

#### Regress gendered on log price controling for category

In [463]:
# Mapping for the 'oos_gender' column
gendered_map = {'Neutral': 'a_Ungendered', 'Male': 'b_Gendered', 'Female': 'b_Gendered'}

# Create a new column 'gender' based on 'oos_gender'
merged_df_prices_unit['gendered'] = merged_df_prices_unit['gender'].map(gendered_map)

In [464]:
reg_lup2_cat = smf.ols('log_price_per_unit ~ gendered + category' , data=merged_df_prices_unit).fit()
print(reg_lup2_cat.summary())

                            OLS Regression Results                            
Dep. Variable:     log_price_per_unit   R-squared:                       0.252
Model:                            OLS   Adj. R-squared:                  0.251
Method:                 Least Squares   F-statistic:                     165.6
Date:                Sun, 30 Mar 2025   Prob (F-statistic):               0.00
Time:                        15:18:52   Log-Likelihood:                -8335.1
No. Observations:                5410   AIC:                         1.669e+04
Df Residuals:                    5398   BIC:                         1.677e+04
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

#### Regress gender on log price controling for category with category/gender interactions

In [274]:
reg_lup_intcat = smf.ols('log_price_per_unit ~ gender * category' , data=merged_df_prices_unit).fit()
print(reg_lup_intcat.summary())

                            OLS Regression Results                            
Dep. Variable:     log_price_per_unit   R-squared:                       0.285
Model:                            OLS   Adj. R-squared:                  0.281
Method:                 Least Squares   F-statistic:                     76.52
Date:                Sat, 29 Mar 2025   Prob (F-statistic):               0.00
Time:                        18:52:07   Log-Likelihood:                -8215.2
No. Observations:                5410   AIC:                         1.649e+04
Df Residuals:                    5381   BIC:                         1.668e+04
Df Model:                          28                                         
Covariance Type:            nonrobust                                         
                                                        coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------

#### Create coefficient plot of above regression

In [474]:
# Prepare data (same extraction as before)
coefs = reg_lup_intcat.params
errors = reg_lup_intcat.bse
interaction_terms = coefs[coefs.index.str.contains("gender")]

coef_df = pd.DataFrame({
    "Variable": interaction_terms.index,
    "Coefficient": interaction_terms.values,
    "SE": errors[interaction_terms.index]
})

# Extract components
coef_df[["Gender", "Category"]] = coef_df["Variable"].str.extract(r'gender\[T\.(.*?)\]:category\[T\.(.*?)\]')
# Update Gender
coef_df['Gender'] = np.where(coef_df['Variable'] == 'gender[T.Male]', 'Male',
                              np.where(coef_df['Variable'] == 'gender[T.Neutral]', 'Neutral', coef_df['Gender']))

# Update Category
coef_df['Category'] = np.where(coef_df['Variable'] == 'gender[T.Male]', 'Body Wash',
                                np.where(coef_df['Variable'] == 'gender[T.Neutral]', 'Body Wash', coef_df['Category']))
coef_df["CI_lower"] = coef_df["Coefficient"] - 1.96*coef_df["SE"]
coef_df["CI_upper"] = coef_df["Coefficient"] + 1.96*coef_df["SE"]

# Create numerical y-position with manual dodging
categories = coef_df["Category"].unique()
category_map = {cat:i for i,cat in enumerate(categories)}
coef_df["y_pos"] = coef_df["Category"].map(category_map)
coef_df["y_pos"] += coef_df["Gender"].map({"Male": 0.25, "Neutral": -0.25})  # Dodge offset

# Create figure
fig = go.Figure()

# Add alternating background
for i, cat in enumerate(categories):
    fig.add_shape(
        type="rect",
        x0=-5, x1=2,
        y0=i-0.5, y1=i+0.5,
        fillcolor='lightgrey' if i % 2 == 0 else "grey",
        opacity=0.2,
        layer="below",
        line_width=0
    )

# Add Male points and error bars
male = coef_df[coef_df["Gender"] == "Male"]
fig.add_trace(go.Scatter(
    x=male["Coefficient"],
    y=male["y_pos"],
    mode="markers",
    name="Male",
    marker=dict(color="#1f77b4", size=12),
    error_x=dict(
        array=male["SE"],
        color="black",
        thickness=1
    ),
    customdata=male[["Category", "Coefficient", "CI_lower", "CI_upper"]],
    hovertemplate=(
        "<b>Male</b><br>"
        "Category: %{customdata[0]}<br>"
        "Estimate: %{customdata[1]:.2f}<br>"
        "95%% CI: [%{customdata[2]:.2f}, %{customdata[3]:.2f}]<extra></extra>"
    )
))

# Add Neutral points and error bars
neutral = coef_df[coef_df["Gender"] == "Neutral"]
fig.add_trace(go.Scatter(
    x=neutral["Coefficient"],
    y=neutral["y_pos"],
    mode="markers",
    name="Neutral",
    marker=dict(color="#2ca02c", size=12),
    error_x=dict(
        array=neutral["SE"],
        color="black",
        thickness=1
    ),
    customdata=neutral[["Category", "Coefficient", "CI_lower", "CI_upper"]],
    hovertemplate=(
        "<b>Neutral</b><br>"
        "Category: %{customdata[0]}<br>"
        "Estimate: %{customdata[1]:.2f}<br>"
        "95%% CI: [%{customdata[2]:.2f}, %{customdata[3]:.2f}]<extra></extra>"
    )
))

# Add zero line
fig.add_vline(
    x=0, line_width=1, line_dash="solid", 
    line_color="red", opacity=0.7
)

# Final layout
fig.update_layout(
    title="Interaction Coefficients (Gender × Category)",
    xaxis_title="Coefficient Estimate",
    yaxis_title="",
    xaxis=dict(range=[-5, 2], dtick=1),
    yaxis=dict(
        tickvals=list(range(len(categories))),
        ticktext=categories,
        range=[-0.5, len(categories)-0.5]
    ),
    height=600,
    width=800,
    plot_bgcolor="white",
    showlegend=True
)

fig.show()

#### Regress gender on overall discounts controlling for category

In [180]:
reg_discount = smf.ols('discount ~ gender + category' , data=merged_df_prices).fit()
print(reg_discount.summary())

                            OLS Regression Results                            
Dep. Variable:               discount   R-squared:                       0.028
Model:                            OLS   Adj. R-squared:                  0.026
Method:                 Least Squares   F-statistic:                     15.14
Date:                Sat, 29 Mar 2025   Prob (F-statistic):           7.13e-37
Time:                        14:14:44   Log-Likelihood:                 5969.8
No. Observations:                7507   AIC:                        -1.191e+04
Df Residuals:                    7492   BIC:                        -1.181e+04
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

#### Regress gender on loyalty discounts controling for category

In [181]:
reg_ndiscount = smf.ols('loyalty_discount ~ gender + category' , data=merged_df_prices).fit()
print(reg_ndiscount.summary())

                            OLS Regression Results                            
Dep. Variable:       loyalty_discount   R-squared:                       0.191
Model:                            OLS   Adj. R-squared:                  0.154
Method:                 Least Squares   F-statistic:                     5.133
Date:                Sat, 29 Mar 2025   Prob (F-statistic):           3.20e-07
Time:                        14:15:06   Log-Likelihood:                 241.41
No. Observations:                 251   AIC:                            -458.8
Df Residuals:                     239   BIC:                            -416.5
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

#### Regress gender on normal discount controling for product category

In [182]:
reg_ndiscount = smf.ols('normal_discount ~ gender + category' , data=merged_df_prices).fit()
print(reg_ndiscount.summary())

                            OLS Regression Results                            
Dep. Variable:        normal_discount   R-squared:                       0.477
Model:                            OLS   Adj. R-squared:                  0.471
Method:                 Least Squares   F-statistic:                     80.22
Date:                Sat, 29 Mar 2025   Prob (F-statistic):          5.02e-162
Time:                        14:15:17   Log-Likelihood:                 746.99
No. Observations:                1245   AIC:                            -1464.
Df Residuals:                    1230   BIC:                            -1387.
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

#### Discount visualization by category

In [433]:
# Calculate both mean discount and count of discounted items
discount_cat = (merged_df_prices[merged_df_prices['discount'] > 0]
                 .groupby(['gender','category'])
                 .agg(mean_discount=('discount', 'mean'),
                      count=('discount', 'count'))
                 .reset_index())

In [434]:
# Define custom colors for gender
gender_colors = {"Female": '#ff69b4', "Male": "#1f77b4", "Neutral": "#2ca02c"}

# Create the grouped bar chart with count labels
bars = alt.Chart(discount_cat).mark_bar().encode(
    x=alt.X("category:N", axis=alt.Axis(title=None, labelAngle=-45)),
    y=alt.Y("mean_discount:Q", title="Average Discount", axis=alt.Axis(format=".0%", titleAngle=0, titleY=-10)),
    color=alt.Color("gender:N", 
                  scale=alt.Scale(domain=list(gender_colors.keys()), 
                                range=list(gender_colors.values())),
    legend=alt.Legend(title="Gender")),
    xOffset="gender:N"
).properties(
    width=600,
    title={
        "text": "Average Discounts by Category and Gender",
    }
)

# Add count labels above bars
count_labels = alt.Chart(discount_cat).mark_text(
    align='center',
    baseline='bottom',
    dy=-5,  # Adjust this to position the label above the bar
    fontSize=9
).encode(
    x=alt.X("category:N"),
    y="mean_discount:Q",
    xOffset="gender:N",
    text=alt.Text("count:Q", format=".0f"),
    color=alt.value("black")
)

# Add explanatory annotation
annotation = alt.Chart(pd.DataFrame({'text': ['Note: Numbers above bars show sample sizes']})).mark_text(
    align='left',
    baseline='top',
    fontSize=10,
    color='gray'
).encode(
    text='text:N'
).properties(
    width=600
)

# Combine all elements
chart = alt.vconcat(
    bars + count_labels,
    annotation
).configure_view(
    stroke='transparent'
).configure_axis(
    grid=False
).configure_title(
    anchor='middle',
    subtitleFontSize=11,
    subtitlePadding=5
)

chart

alt.VConcatChart(...)

#### Discount visualization by type

In [438]:
# Calculate both mean discount and count of discounted items
discount_type = (merged_df_prices[merged_df_prices['discount'] > 0]
                 .groupby(['gender'])
                 .agg(mean_discount=('discount', 'mean'),
                      mean_ldiscount=('loyalty_discount', 'mean'),
                      mean_ndiscount=('normal_discount', 'mean'),
                      count=('discount', 'count'),
                      lcount=('loyalty_discount', 'count'),
                      ncount=('normal_discount', 'count'))
                 .reset_index())

discount_type_a = discount_type[['gender','mean_discount','mean_ldiscount','mean_ndiscount']].melt(id_vars='gender', 
                         var_name='Discount Type', 
                         value_name='Discount Value')
discount_type_c = discount_type[['gender','count','lcount','ncount']].melt(id_vars='gender', 
                         var_name='Discount Type', 
                         value_name='Discount Count')
discount_type = pd.concat([discount_a,discount_count],axis=1).iloc[:,2:]

In [439]:
discount_type['Discount Type'] = discount_type['Discount Type'].replace({
    'count': 'Overall Discount',
    'lcount': 'Loyalty Discount',
    'ncount': 'Normal Discount'
})

# Define custom colors for gender
gender_colors = {"Female": '#ff69b4', "Male": "#1f77b4", "Neutral": "#2ca02c"}

# Create the grouped bar chart with count labels
bars = alt.Chart(discount_type).mark_bar().encode(
    x=alt.X("Discount Type:N", axis=alt.Axis(title=None, labelAngle=0)),
    y=alt.Y("Discount Value:Q", title="Average Discount", axis=alt.Axis(format=".0%", titleAngle=0, titleY=-10)),
    color=alt.Color("gender:N", 
                  scale=alt.Scale(domain=list(gender_colors.keys()), 
                                range=list(gender_colors.values())),
    legend=alt.Legend(title="Gender")),
    xOffset="gender:N"
).properties(
    width=600,
    title={
        "text": "Average Discounts by Type and Gender",
    }
)

# Add count labels above bars
count_labels = alt.Chart(discount_type).mark_text(
    align='center',
    baseline='bottom',
    dy=-5,  # Adjust this to position the label above the bar
    fontSize=9
).encode(
    x=alt.X("Discount Type:N"),
    y="Discount Value:Q",
    xOffset="gender:N",
    text=alt.Text("Discount Count:Q", format=".0f"),
    color=alt.value("black")
)

# Add explanatory annotation
annotation = alt.Chart(pd.DataFrame({'text': ['Note: Numbers above bars show sample sizes']})).mark_text(
    align='left',
    baseline='top',
    fontSize=10,
    color='gray'
).encode(
    text='text:N'
).properties(
    width=600
)

# Combine all elements
discount_type_chart = alt.vconcat(
    bars + count_labels,
    annotation
).configure_view(
    stroke='transparent'
).configure_axis(
    grid=False
).configure_title(
    anchor='middle',
    subtitleFontSize=11,
    subtitlePadding=5
)

discount_type_chart

alt.VConcatChart(...)